In [ ]:
# @title LLM Prover
from IPython.display import HTML, display
display(HTML('''
<style>
.pysolvr-header { background: linear-gradient(135deg, #8B5CF622, #06B6D422); border: 1px solid #8B5CF644; border-radius: 12px; padding: 24px; text-align: center; font-family: Inter, system-ui, sans-serif; color: #f1f5f9; }
.pysolvr-header h1 { margin: 0; font-size: 28px; font-weight: 300; }
.pysolvr-header p { margin: 8px 0 0; color: #94a3b8; font-size: 14px; }
.pysolvr-badge { display: inline-block; padding: 2px 8px; border-radius: 4px; font-size: 11px; font-weight: 500; background: #8B5CF622; color: #8B5CF6; }
</style>
<div class="pysolvr-header">
    <h1>LLM Prover</h1>
    <p>Your AI truth layer - benchmark, monitor, and prove how every model performs on YOUR data</p>
    <p style="margin-top:12px"><span class="pysolvr-badge">v0.2.0+t0.3.0</span></p>
</div>
'''))


In [ ]:
# @title Setup
!pip install -q pysolvr-client
import sys
from IPython.display import HTML, display
from google.colab import userdata, drive

try:
    API_KEY = userdata.get('LLM_PROVER_API_KEY')
except userdata.SecretNotFoundError:
    display(HTML('''
    <div style="background:#ef444422;border:1px solid #ef444444;border-radius:8px;padding:16px;font-family:Inter,system-ui,sans-serif;color:#f1f5f9">
        <p style="margin:0 0 8px;font-weight:500">API key not found in Colab Secrets</p>
        <ol style="color:#94a3b8;font-size:13px;margin:0;padding-left:20px">
            <li>Click the <b>key icon</b> in the left sidebar</li>
            <li>Add a new secret named: <code>LLM_PROVER_API_KEY</code></li>
            <li>Paste your API key as the value</li>
            <li>Toggle <b>Notebook access</b> ON</li>
            <li>Re-run this cell</li>
        </ol>
    </div>
    '''))
    assert False, 'Setup incomplete - add your API key to Colab Secrets first'

drive.mount('/content/drive', force_remount=False)

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://s8xtlhju90.execute-api.us-east-1.amazonaws.com/dev'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#8B5CF6', '#06B6D4')
drive_mgr = DriveManager('llmprover', ['comparisons', 'benchmarks', 'reports', 'evaluations'])
drive_mgr.ensure_folders()

if client.health_check():
    ui.success('Connected to LLM Prover API', f'Drive: {drive_mgr.root}')
else:
    ui.error('Could not connect', 'API key may be invalid or expired')


In [ ]:
# @title Compare Models
# @markdown Test your prompt across multiple LLMs in parallel
PROMPT = ""  # @param {type:"string"}
CONTEXT = ""  # @param {type:"string"}
MAX_TOKENS = 1000  # @param {type:"integer"}

if not PROMPT.strip():
    ui.error('Enter a prompt above and re-run this cell')
else:
    payload = {'prompt': PROMPT, 'params': {'max_tokens': MAX_TOKENS, 'temperature': 0.7}}
    if CONTEXT.strip():
        payload['context'] = CONTEXT
    result = client.run_async('/compare', payload)
    if result['ok']:
        d = result['data']
        rows = []
        for r in d.get('results', []):
            preview = (r.get('result_text') or '')[:120]
            if r.get('error'):
                preview = f"ERROR: {r['error'][:80]}"
            rows.append({'Provider': r['provider'], 'Model': r.get('model',''), 'Latency': f"{r.get('latency_ms',0)}ms", 'Cost': f"${r.get('cost_usd',0):.4f}", 'Output': preview})
        results_html = '<table class="pysolvr-table"><thead><tr>' + ''.join(f'<th>{c}</th>' for c in ['Provider','Model','Latency','Cost','Output']) + '</tr></thead><tbody>'
        for row in rows:
            results_html += '<tr>' + ''.join(f'<td>{row[c]}</td>' for c in ['Provider','Model','Latency','Cost','Output']) + '</tr>'
        results_html += '</tbody></table>'
        enr = d.get('enrichments', {})
        enr_html = f"<p><b>Recommendation:</b> {enr.get('recommendation','N/A')}</p>"
        enr_html += f"<p><b>Fastest:</b> {enr.get('fastest','N/A')} | <b>Cheapest:</b> {enr.get('cheapest','N/A')}</p>"
        if enr.get('agreement_score') is not None:
            enr_html += f"<p><b>Agreement:</b> {enr['agreement_score']:.0%}</p>"
        full_html = ''
        for r in d.get('results', []):
            text = r.get('result_text') or r.get('error') or 'No output'
            full_html += f"<details><summary><b>{r['provider']}</b> ({r.get('model','')})</summary><pre style='white-space:pre-wrap;color:#94a3b8;font-size:12px;margin:8px 0'>{text}</pre></details>"
        tabs = {'Results': results_html, 'Analysis': enr_html, 'Full Outputs': full_html}
        ui.tabs(tabs)
        ui.cost_badge(sum(r.get('tokens_total',0) for r in d.get('results',[])), d.get('total_cost_usd',0))
        import json as _json
        cid = d['comparison_id']
        drive_mgr.save('comparisons', f'{cid}.json', _json.dumps(d, indent=2), meta={'prompt': PROMPT, 'total_cost_usd': d.get('total_cost_usd',0), 'providers': d.get('providers_responded',[])})
        ui.success(f'Saved to Drive: comparisons/{cid}.json')
    else:
        ui.error(result.get('error', 'Compare failed'), result.get('upgrade_hint', 'Check your plan and balance'))


In [ ]:
# @title Verdictator
# @markdown Score and rank model responses against your expected output
PROMPT = ""  # @param {type:"string"}
CONTEXT = ""  # @param {type:"string"}
EXPECTED_OUTPUT = ""  # @param {type:"string"}
PROVIDERS = "All"  # @param ["All", "Grok", "Anthropic", "OpenAI"]
MAX_TOKENS = 1000  # @param {type:"integer"}

if not PROMPT.strip() or not CONTEXT.strip() or not EXPECTED_OUTPUT.strip():
    ui.error('Fill in PROMPT, CONTEXT, and EXPECTED_OUTPUT above, then re-run')
else:
    payload = {
        'prompt': PROMPT,
        'context': CONTEXT,
        'mode': 'gold_standard',
        'expected_output': EXPECTED_OUTPUT,
        'params': {'max_tokens': MAX_TOKENS, 'temperature': 0.7}
    }
    if PROVIDERS != 'All':
        payload['providers'] = [PROVIDERS.lower()]
    result = client.run_async('/evaluate', payload)
    if result['ok']:
        d = result['data']
        # Scores table
        rows = []
        for r in d.get('results', []):
            rows.append({
                'Provider': r['provider'],
                'Model': r.get('model', ''),
                'Quality': f"{r.get('quality_score', 0):.1f}",
                'Value': f"{r.get('value_score', 0):.0f}",
                'Cost': f"${r.get('cost_usd', 0):.4f}",
                'Latency': f"{r.get('latency_ms', 0)}ms"
            })
        # Winner summary
        w = d.get('winner', {})
        winner_html = f"<p><b>Best quality:</b> {w.get('by_quality', 'N/A')} | <b>Best value:</b> {w.get('by_value', 'N/A')}</p>"
        if w.get('reasoning'):
            winner_html += f"<p style='color:#94a3b8;font-size:12px'>{w['reasoning']}</p>"
        # Breakdown per model
        breakdown_html = ''
        for r in d.get('results', []):
            bd = r.get('breakdown', {})
            if bd:
                breakdown_html += f"<details><summary><b>{r['provider']}</b> (quality: {r.get('quality_score',0):.1f})</summary>"
                breakdown_html += '<ul>' + ''.join(f"<li>{k}: {v:.1f}</li>" for k, v in bd.items()) + '</ul></details>'
        # Full responses
        responses_html = ''
        for r in d.get('results', []):
            text = r.get('response_text', 'No output')
            responses_html += f"<details><summary><b>{r['provider']}</b></summary><pre style='white-space:pre-wrap;color:#94a3b8;font-size:12px'>{text}</pre></details>"
        # Build table HTML
        cols = ['Provider', 'Model', 'Quality', 'Value', 'Cost', 'Latency']
        table_html = '<table class="pysolvr-table"><thead><tr>' + ''.join(f'<th>{c}</th>' for c in cols) + '</tr></thead><tbody>'
        for row in rows:
            table_html += '<tr>' + ''.join(f'<td>{row[c]}</td>' for c in cols) + '</tr>'
        table_html += '</tbody></table>'
        tabs = {'Scores': table_html + winner_html, 'Breakdown': breakdown_html or '<p>No breakdown available</p>', 'Responses': responses_html}
        judge_used = d.get('judge_used', False)
        if judge_used:
            tabs['Scores'] += '<p style="font-size:11px;color:#94a3b8">Judge scoring applied (Pro+)</p>'
        ui.tabs(tabs)
        ui.cost_badge(0, d.get('total_cost_usd', 0))
        # Save to Drive
        import json as _json
        eid = d.get('evaluation_id', d.get('collection_hash', 'unknown'))
        drive_mgr.save('evaluations', f'{eid}.json', _json.dumps(d, indent=2), meta={'prompt': PROMPT, 'mode': 'gold_standard', 'collection_hash': d.get('collection_hash', ''), 'total_cost_usd': d.get('total_cost_usd', 0)})
        ui.success(f'Saved to Drive: evaluations/{eid}.json')
    else:
        ui.error(result.get('error', 'Evaluation failed'), result.get('upgrade_hint', 'Check your plan and balance'))


In [ ]:
# @title Available Models
result = client.call('GET', '/models')
if result['ok']:
    rows = []
    for pname, pdata in result['data'].get('providers', {}).items():
        for m in pdata.get('models', []):
            if m.get('status') == 'eol':
                continue
            default = ' (default)' if m.get('id') == pdata.get('default') else ''
            rows.append({'Provider': pname, 'Model': m['id'] + default, 'Input $/1K': f"${m.get('cost_per_1k_input','?')}", 'Output $/1K': f"${m.get('cost_per_1k_output','?')}", 'Max Tokens': m.get('max_tokens','')})
    ui.table(rows, ['Provider', 'Model', 'Input $/1K', 'Output $/1K', 'Max Tokens'])
else:
    ui.error(result.get('error', 'Could not fetch models'))


In [ ]:
# @title Account
result = client.call('GET', '/account')
if result['ok']:
    d = result['data']
    plan = d.get('plan', 'free').title()
    spend = d.get('monthly_spend_usd') or 0
    limit = d.get('monthly_limit_usd') or 0
    ui.card('Account', f"<b>Plan:</b> {plan}<br><b>Spend this month:</b> ${spend:.2f} / ${limit:.2f}", status='success')
else:
    ui.error(result.get('error', 'Could not fetch account'), 'Check your API key')


In [ ]:
# @title Usage
result = client.call('GET', '/account')
if result['ok']:
    d = result['data']
    spent = d.get('monthly_spend_usd') or 0
    limit = d.get('monthly_limit_usd') or 1
    ui.usage_bar(spent, limit)
else:
    ui.error(result.get('error', 'Could not fetch usage'), 'Check your API key')


In [ ]:
# @title Support
from IPython.display import HTML, display
display(HTML('''
<style>
.pysolvr-help details { margin: 8px 0; font-family: Inter, system-ui, sans-serif; color: #f1f5f9; }
.pysolvr-help summary { cursor: pointer; font-weight: 500; font-size: 14px; }
.pysolvr-help p, .pysolvr-help li { color: #94a3b8; font-size: 13px; }
</style>
<div class="pysolvr-help">
<details><summary>Quick Start</summary>
<ol><li>Run Setup with your API key</li><li>Use Compare Models to test prompts across providers</li><li>Results save automatically to Google Drive</li></ol>
</details>
<details><summary>Where are my files?</summary>
<p>Google Drive > My Drive > pysolvr > llmprover > comparisons/</p>
</details>
<details><summary>What models are available?</summary>
<p>Run the "Available Models" cell to see current providers, models, and pricing.</p>
</details>
<details><summary>Need help?</summary>
<p>Email support@pysolvr.com or open a ticket using the Support cell.</p>
</details>
</div>
'''))
